In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))

In [ ]:
import sys
print(sys.executable)
print(sys.version)

/Users/apple/dev/EvoFinGraph/.venv/bin/python
3.13.7 (main, Aug 14 2025, 11:12:11) [Clang 17.0.0 (clang-1700.0.13.3)]


In [ ]:
from src.models.base import BaseModel

In [ ]:
BaseModel

src.models.base.BaseModel

In [ ]:
import importlib

import src.config
import src.pipeline.builder
import src.graph.converter
import src.models.graphsage

importlib.reload(src.pipeline.builder)
importlib.reload(src.graph.converter)
importlib.reload(src.models.graphsage)

from src.pipeline.builder import EvoFinGraphPipeline
from src.graph.converter import GraphConverter
from src.models.graphsage import GraphSAGEModel

In [ ]:
pipeline = EvoFinGraphPipeline()

pipeline.build()

2026-07-20 13:39:33,385 | INFO | Loading processed dataset...
2026-07-20 13:39:45,513 | INFO | Dataset loaded successfully.
2026-07-20 13:39:45,515 | INFO | Building temporal graphs...
2026-07-20 13:39:45,515 | INFO | Building graph for timestep 20
2026-07-20 13:39:45,700 | INFO | Graph built successfully | Nodes: 900 | Edges: 613
2026-07-20 13:39:45,700 | INFO | Snapshot 20 validated.
2026-07-20 13:39:45,752 | INFO | Detected 327 communities.
2026-07-20 13:39:45,754 | INFO | Building graph for timestep 21
2026-07-20 13:39:45,891 | INFO | Graph built successfully | Nodes: 641 | Edges: 518
2026-07-20 13:39:45,892 | INFO | Snapshot 21 validated.
2026-07-20 13:39:45,922 | INFO | Detected 238 communities.
2026-07-20 13:39:45,924 | INFO | Building graph for timestep 22
2026-07-20 13:39:46,292 | INFO | Graph built successfully | Nodes: 1763 | Edges: 1537
2026-07-20 13:39:46,293 | INFO | Snapshot 22 validated.
2026-07-20 13:39:46,394 | INFO | Detected 536 communities.
2026-07-20 13:39:46,397 

In [ ]:
pipeline.summary()

EVOFINGRAPH PIPELINE
Snapshots              : 5
Tracked Communities    : 1037
Feature Matrix Shape   : (528, 16)


In [ ]:
feature_matrix = pipeline.feature_matrix

feature_matrix.head()

,pcid,timestep,community,growth_rate,density_change,degree_change,fraud_growth,clustering_change,feature_drift,variance_drift,lifetime,persistence,average_match_score,node_churn,structural_stability,is_fraud_community
0,6,21,11,-0.151515,0.178571,-0.043478,1.357143,0.0,4.610801,0.295710,2,0.4,0.859100,1.848485,0.931084,1
1,11,21,18,0.666667,-0.169231,0.530612,-1.000000,0.0,4.278830,0.730030,2,0.4,0.903350,2.666667,0.810845,0
2,15,21,9,0.000000,0.000000,0.000000,0.000000,0.0,9.080104,0.000000,3,0.6,0.915467,2.000000,1.000000,0
3,15,22,457,0.000000,0.000000,0.000000,0.000000,0.0,4.331653,0.000000,3,0.6,0.915467,2.000000,1.000000,0
4,18,21,87,-0.391304,0.642857,0.008117,0.000000,0.0,3.258021,0.064171,4,0.8,0.793975,1.608696,0.821699,1


In [ ]:
from src.models.data import ModelData

X_train, X_test, y_train, y_test = ModelData.prepare(
    feature_matrix
)

In [ ]:
"""Train logistic regression model"""
from src.models.logistic import LogisticModel

model = LogisticModel()

model.fit(
    X_train,
    y_train,
)

In [ ]:
importance = model.feature_importance(
    X_train.columns
)

importance

{'fraud_growth': np.float64(1.1035315533366545),
 'density_change': np.float64(0.6986070380543582),
 'structural_stability': np.float64(-0.47786163297257883),
 'feature_drift': np.float64(-0.38054042777938185),
 'degree_change': np.float64(0.24794286441997868),
 'node_churn': np.float64(0.08202755449643861),
 'growth_rate': np.float64(0.06628974916498764),
 'variance_drift': np.float64(0.0638143691491139),
 'clustering_change': np.float64(-0.014967009727762057)}

EVALUATION FOLDER'S METRICS TESTING

In [ ]:
from src.evaluation.metrics import EvaluationMetrics

predictions = model.predict(
    X_test
)

probabilities = model.predict_proba(
    X_test
)

metrics = EvaluationMetrics.compute(

    y_test,

    predictions,

    probabilities,

)

metrics

{'accuracy': 0.6698113207547169,
 'precision': 0.3877551020408163,
 'recall': 0.7916666666666666,
 'f1': 0.5205479452054794,
 'roc_auc': 0.8724593495934958}

In [ ]:
from src.evaluation.report import EvaluationReport

In [ ]:
report = EvaluationReport()

In [ ]:
report.add(
    "Logistic Regression",
    metrics,
)

In [ ]:
report.summary()

MODEL COMPARISON
                 model  accuracy  precision    recall        f1   roc_auc
0  Logistic Regression  0.669811   0.387755  0.791667  0.520548  0.872459


,model,accuracy,precision,recall,f1,roc_auc
0,Logistic Regression,0.669811,0.387755,0.791667,0.520548,0.872459


XGBOOST TESTING

In [ ]:
from src.models.xgboost import XGBoostModel

xgb = XGBoostModel()

xgb.fit(
    X_train,
    y_train,
)

In [ ]:
"""Predict"""
xgb_predictions = xgb.predict(
    X_test
)

xgb_probabilities = xgb.predict_proba(
    X_test
)

In [ ]:
from src.evaluation.metrics import EvaluationMetrics

xgb_metrics = EvaluationMetrics.compute(

    y_test,

    xgb_predictions,

    xgb_probabilities,

)

xgb_metrics

{'accuracy': 0.839622641509434,
 'precision': 0.6666666666666666,
 'recall': 0.5833333333333334,
 'f1': 0.6222222222222222,
 'roc_auc': 0.853658536585366}

In [ ]:
importance = xgb.feature_importance(
    X_train.columns
)

importance

{'feature_drift': np.float32(0.237506),
 'variance_drift': np.float32(0.13049889),
 'fraud_growth': np.float32(0.12957232),
 'density_change': np.float32(0.12261081),
 'structural_stability': np.float32(0.10358988),
 'growth_rate': np.float32(0.095519386),
 'degree_change': np.float32(0.09538253),
 'node_churn': np.float32(0.08532023),
 'clustering_change': np.float32(0.0)}

In [ ]:
logistic_predictions = model.predict(
    X_test
)

logistic_probabilities = model.predict_proba(
    X_test
)

In [ ]:
from src.evaluation.metrics import EvaluationMetrics

logistic_metrics = EvaluationMetrics.compute(

    y_test,

    logistic_predictions,

    logistic_probabilities,

)

logistic_metrics

{'accuracy': 0.6698113207547169,
 'precision': 0.3877551020408163,
 'recall': 0.7916666666666666,
 'f1': 0.5205479452054794,
 'roc_auc': 0.8724593495934958}

In [ ]:
from src.evaluation.report import EvaluationReport

report = EvaluationReport()

report.add(

    "Logistic Regression",

    logistic_metrics,

)

report.add(

    "XGBoost",

    xgb_metrics,

)

comparison = report.summary()

comparison

MODEL COMPARISON
                 model  accuracy  precision    recall        f1   roc_auc
1              XGBoost  0.839623   0.666667  0.583333  0.622222  0.853659
0  Logistic Regression  0.669811   0.387755  0.791667  0.520548  0.872459


,model,accuracy,precision,recall,f1,roc_auc
1,XGBoost,0.839623,0.666667,0.583333,0.622222,0.853659
0,Logistic Regression,0.669811,0.387755,0.791667,0.520548,0.872459


In [ ]:
report.rank("f1")

,model,accuracy,precision,recall,f1,roc_auc
1,XGBoost,0.839623,0.666667,0.583333,0.622222,0.853659
0,Logistic Regression,0.669811,0.387755,0.791667,0.520548,0.872459


In [ ]:
import pandas as pd

comparison = pd.DataFrame({

    "Logistic": model.feature_importance(
        X_train.columns
    ),

    "XGBoost": xgb.feature_importance(
        X_train.columns
    ),

})

comparison

,Logistic,XGBoost
fraud_growth,1.103532,0.129572
density_change,0.698607,0.122611
structural_stability,-0.477862,0.103590
feature_drift,-0.380540,0.237506
degree_change,0.247943,0.095383
node_churn,0.082028,0.085320
growth_rate,0.066290,0.095519
variance_drift,0.063814,0.130499
clustering_change,-0.014967,0.000000


In [ ]:
"""testing graph converter to implement graphsage"""
from src.graph.converter import GraphConverter

graph = pipeline.get_graph(20)

data = GraphConverter.to_pyg(graph)

data

/Users/apple/dev/EvoFinGraph/src/graph/converter.py:115: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /Users/runner/work/pytorch/pytorch/torch/csrc/utils/tensor_new.cpp:256.)
  x = torch.tensor(


Data(
  x=[900, 165],
  edge_index=[2, 1226],
  y=[900],
  community=[900],
  node_mapping={
    209556175.0=0,
    52934000.0=1,
    37574467.0=2,
    209972191.0=3,
    73339227.0=4,
    209837677.0=5,
    5286455.0=6,
    209826974.0=7,
    210644361.0=8,
    25642291.0=9,
    209952987.0=10,
    209952974.0=11,
    211086336.0=12,
    209724105.0=13,
    209045031.0=14,
    210640906.0=15,
    209681259.0=16,
    209118488.0=17,
    209794811.0=18,
    209487402.0=19,
    209678521.0=20,
    209533787.0=21,
    209918381.0=22,
    209537331.0=23,
    209905828.0=24,
    210192146.0=25,
    1275118.0=26,
    4659656.0=27,
    209091416.0=28,
    209692891.0=29,
    107697436.0=30,
    209518228.0=31,
    211078002.0=32,
    209688652.0=33,
    209921231.0=34,
    209824867.0=35,
    209673949.0=36,
    209842048.0=37,
    8182823.0=38,
    209798865.0=39,
    1552384.0=40,
    209818962.0=41,
    209045727.0=42,
    209674270.0=43,
    209892725.0=44,
    58114564.0=45,
    20892789

In [ ]:
print(data.community.shape)

print(data.timestep)

print(len(data.node_mapping))

print(data.reverse_mapping[0])

torch.Size([900])
20
900
209556175.0


In [ ]:
import importlib
import src.graph.converter

importlib.reload(src.graph.converter)

<module 'src.graph.converter' from '/Users/apple/dev/EvoFinGraph/src/graph/converter.py'>

In [ ]:
from src.graph.converter import GraphConverter

graph = pipeline.get_graph(20)

data = GraphConverter.to_pyg(graph)

print(data.num_communities)

327


In [ ]:
from src.graph.converter import GraphConverter

pyg_graphs = GraphConverter.convert_all(
    pipeline.graphs
)

In [ ]:
pyg_graphs.keys()

dict_keys([20, 21, 22, 23, 24])

In [ ]:
pyg_graphs[20]

Data(
  x=[900, 165],
  edge_index=[2, 1226],
  y=[900],
  community=[900],
  node_mapping={
    209556175.0=0,
    52934000.0=1,
    37574467.0=2,
    209972191.0=3,
    73339227.0=4,
    209837677.0=5,
    5286455.0=6,
    209826974.0=7,
    210644361.0=8,
    25642291.0=9,
    209952987.0=10,
    209952974.0=11,
    211086336.0=12,
    209724105.0=13,
    209045031.0=14,
    210640906.0=15,
    209681259.0=16,
    209118488.0=17,
    209794811.0=18,
    209487402.0=19,
    209678521.0=20,
    209533787.0=21,
    209918381.0=22,
    209537331.0=23,
    209905828.0=24,
    210192146.0=25,
    1275118.0=26,
    4659656.0=27,
    209091416.0=28,
    209692891.0=29,
    107697436.0=30,
    209518228.0=31,
    211078002.0=32,
    209688652.0=33,
    209921231.0=34,
    209824867.0=35,
    209673949.0=36,
    209842048.0=37,
    8182823.0=38,
    209798865.0=39,
    1552384.0=40,
    209818962.0=41,
    209045727.0=42,
    209674270.0=43,
    209892725.0=44,
    58114564.0=45,
    20892789

In [ ]:
print(pyg_graphs[20].timestep)

print(pyg_graphs[20].num_communities)

print(len(pyg_graphs[20].node_mapping))

20
327
900


GRAPHSAGE TESTING

In [ ]:
import importlib
import src.models.graphsage

importlib.reload(src.models.graphsage)

<module 'src.models.graphsage' from '/Users/apple/dev/EvoFinGraph/src/models/graphsage.py'>

In [ ]:
from src.graph.converter import GraphConverter

pyg_graphs = GraphConverter.convert_all(
    pipeline.graphs
)

graphs = list(
    pyg_graphs.values()
)

In [ ]:
graphs[0]

Data(
  x=[900, 165],
  edge_index=[2, 1226],
  y=[900],
  community=[900],
  node_mapping={
    209556175.0=0,
    52934000.0=1,
    37574467.0=2,
    209972191.0=3,
    73339227.0=4,
    209837677.0=5,
    5286455.0=6,
    209826974.0=7,
    210644361.0=8,
    25642291.0=9,
    209952987.0=10,
    209952974.0=11,
    211086336.0=12,
    209724105.0=13,
    209045031.0=14,
    210640906.0=15,
    209681259.0=16,
    209118488.0=17,
    209794811.0=18,
    209487402.0=19,
    209678521.0=20,
    209533787.0=21,
    209918381.0=22,
    209537331.0=23,
    209905828.0=24,
    210192146.0=25,
    1275118.0=26,
    4659656.0=27,
    209091416.0=28,
    209692891.0=29,
    107697436.0=30,
    209518228.0=31,
    211078002.0=32,
    209688652.0=33,
    209921231.0=34,
    209824867.0=35,
    209673949.0=36,
    209842048.0=37,
    8182823.0=38,
    209798865.0=39,
    1552384.0=40,
    209818962.0=41,
    209045727.0=42,
    209674270.0=43,
    209892725.0=44,
    58114564.0=45,
    20892789

In [ ]:
from importlib import reload
import src.models.graphsage

reload(src.models.graphsage)

from src.models.graphsage import GraphSAGEModel

In [ ]:
from src.graph.converter import GraphConverter

pyg_graphs = GraphConverter.convert_all(
    pipeline.graphs
)

graphs = list(
    pyg_graphs.values()
)

In [ ]:
sage = GraphSAGEModel(

    input_dim=graphs[0].num_node_features,

)

In [ ]:
import time
from types import MethodType

def debug_fit(self, graphs, epochs=5, verbose=True):

    if not isinstance(graphs, (list, tuple)):
        graphs = [graphs]

    self.model.train()

    history = []

    total_graphs = len(graphs)

    overall_start = time.time()

    print("=" * 60)
    print(f"Training on {total_graphs} graph(s)")
    print(f"Epochs: {epochs}")
    print("=" * 60)

    for epoch in range(epochs):

        epoch_start = time.time()

        epoch_loss = 0.0

        print(f"\nEpoch {epoch+1}/{epochs}")

        for graph_idx, data in enumerate(graphs):

            graph_start = time.time()

            self.optimizer.zero_grad()

            logits = self.forward(data)

            loss = self.loss_fn(
                logits,
                data.y,
            )

            loss.backward()

            self.optimizer.step()

            epoch_loss += loss.item()

            print(
                f"  Graph {graph_idx+1}/{total_graphs} | "
                f"Nodes={data.num_nodes:,} | "
                f"Edges={data.edge_index.shape[1]:,} | "
                f"Loss={loss.item():.4f} | "
                f"{time.time()-graph_start:.2f}s"
            )

        epoch_loss /= total_graphs

        history.append(epoch_loss)

        print(
            f"Epoch finished in "
            f"{time.time()-epoch_start:.2f}s | "
            f"Average Loss={epoch_loss:.4f}"
        )

    print("\nTraining Complete!")
    print(f"Total Time: {time.time()-overall_start:.2f}s")

    return history

# Replace only this instance's fit method
sage.fit = MethodType(debug_fit, sage)

In [ ]:
%whos

Variable                 Type                   Data/Info
---------------------------------------------------------
BaseModel                ABCMeta                <class 'src.models.base.BaseModel'>
EvaluationMetrics        type                   <class 'src.evaluation.me<...>trics.EvaluationMetrics'>
EvaluationReport         type                   <class 'src.evaluation.report.EvaluationReport'>
EvoFinGraphPipeline      type                   <class 'src.pipeline.buil<...>der.EvoFinGraphPipeline'>
GraphConverter           type                   <class 'src.graph.converter.GraphConverter'>
GraphSAGEModel           ABCMeta                <class 'src.models.graphsage.GraphSAGEModel'>
LogisticModel            ABCMeta                <class 'src.models.logistic.LogisticModel'>
MethodType               type                   <class 'method'>
ModelData                type                   <class 'src.models.data.ModelData'>
PROJECT_ROOT             PosixPath              /Users/apple/dev/Ev

In [ ]:
print(len(graphs))

5


In [ ]:
for i, graph in enumerate(graphs):

    print(
        i,
        graph.num_nodes,
        graph.edge_index.shape[1],
    )

0 900 1226
1 641 1036
2 1763 3074
3 1187 2092
4 1126 1922


In [ ]:
sage = GraphSAGEModel(

    input_dim=graphs[0].num_node_features,

    hidden_dim=32,   # much faster for now

)

In [ ]:
history = sage.fit(
    graphs[-3:],
    epochs=5,
)

In [ ]:
predictions = sage.predict(
    graphs[-1]
)

probabilities = sage.predict_proba(
    graphs[-1]
)

In [ ]:
from src.evaluation.metrics import EvaluationMetrics

graphsage_metrics = EvaluationMetrics.compute(

    graphs[-1].y.numpy(),

    predictions,

    probabilities[:, 1],

)

graphsage_metrics

{'accuracy': 0.8490230905861457,
 'precision': 0.2676056338028169,
 'recall': 0.1386861313868613,
 'f1': 0.18269230769230768,
 'roc_auc': 0.7556847955244921}

SAGE EVALUATION

In [ ]:
predictions = sage.predict(
    graphs[-1]
)

probabilities = sage.predict_proba(
    graphs[-1]
)

In [ ]:
from src.evaluation.metrics import EvaluationMetrics

graphsage_metrics = EvaluationMetrics.compute(

    graphs[-1].y.numpy(),

    predictions,

    probabilities[:, 1],

)

graphsage_metrics

{'accuracy': 0.8490230905861457,
 'precision': 0.2676056338028169,
 'recall': 0.1386861313868613,
 'f1': 0.18269230769230768,
 'roc_auc': 0.7556847955244921}

In [ ]:
embeddings = sage.embeddings(
    graphs[-1]
)

embeddings.shape

(1126, 32)